In [2]:
import fastf1
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.impute import SimpleImputer


In [5]:
import os
import fastf1

os.makedirs("f1_cache", exist_ok=True)

In [6]:
fastf1.Cache.enable_cache("f1_cache")

# load the f1 dataset (Qutar , 2024)
session_2024= fastf1.get_session(2024,24,"R")
session_2024.load()
laps_2024=session_2024.laps[['Driver','LapTime','Sector1Time','Sector2Time','Sector3Time']].copy()
laps_2024.dropna(inplace=True)

core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No 

In [9]:
laps_2024.head()

,Driver,LapTime,Sector1Time,Sector2Time,Sector3Time
1,VER,0 days 00:01:54.938000,0 days 00:00:18.209000,0 days 00:00:54.569000,0 days 00:00:42.160000
2,VER,0 days 00:01:38.051000,0 days 00:00:23.196000,0 days 00:00:41.997000,0 days 00:00:32.858000
3,VER,0 days 00:01:29.504000,0 days 00:00:18.166000,0 days 00:00:38.187000,0 days 00:00:33.151000
4,VER,0 days 00:01:29.813000,0 days 00:00:18.056000,0 days 00:00:37.945000,0 days 00:00:33.812000
5,VER,0 days 00:01:29.412000,0 days 00:00:18.012000,0 days 00:00:38.101000,0 days 00:00:33.299000


In [11]:
laps_2024.shape

(1014, 5)

In [13]:
laps_2024['Driver'].value_counts()

Driver
VER    57
GAS    57
ALO    57
LEC    57
HUL    57
HAM    57
PIA    57
RUS    57
SAI    57
NOR    57
MAG    56
STR    56
ZHO    56
ALB    56
TSU    56
DOO    56
LAW    54
BOT    29
COL    25
Name: count, dtype: int64

In [14]:
laps_2024["Driver"].unique()

array(['VER', 'GAS', 'ALO', 'LEC', 'STR', 'MAG', 'TSU', 'ALB', 'ZHO',
       'HUL', 'LAW', 'NOR', 'COL', 'HAM', 'SAI', 'DOO', 'RUS', 'BOT',
       'PIA'], dtype=object)

In [ ]:
sector_times_2024=laps_2024.groupby("Driver").agg(
    {
        "Sector1Time (s)" : "mean",
        "Sector2Time (s)" : "mean",
        "Sector3Time (s)" : "mean",
    }
).rest_index()